# ⚡ EfficientNet-B1 — 5-Class Dark Skin Disease
**Melanoma | Basal_Cell_Carcinoma | Tinea | Eczema | Keratosis**

Changes vs 9-class B1:
- dropout 0.3 → 0.35
- drop_path_rate 0.1 → 0.15
- weight_decay 1e-4 → 2e-4
- freeze_epochs 5 → 6
- patience 7 → 6
- Loss boosts: Tinea 3x, Eczema 1.5x (smallest classes)
- Melanoma + BCC sensitivity targets: 95% and 90%

Applied fixes:
- Head: BatchNorm1d → LayerNorm, SiLU → GELU (batch-size-1 safe; consistent with ConvNeXt/Swin)
- Checkpoint criterion: composite score (0.4×mel_sens + 0.3×bcc_sens + 0.3×val_acc) instead of val_acc alone
- melanoma_sensitivity tracked in history and saved to checkpoint

## 1. Install

In [ ]:
%%capture
!pip install timm scikit-learn
print('Done')

## 2. Imports & Reproducibility

In [ ]:
import os, random, warnings, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, roc_curve, auc, confusion_matrix
)
from sklearn.preprocessing import label_binarize
import seaborn as sns

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
# deterministic=True disabled: kills ~25% throughput on H100 by blocking fastest cudnn kernels
# torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True  # fastest kernels for fixed 224×224 input
    print('TF32 + cudnn.benchmark enabled')

## 3. Mount Drive & Copy to SSD

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = Path('/content/drive/MyDrive/IPD/IPD_Datasets_Final_Split')
LOCAL_ROOT = Path('/content/dataset')
SAVE_DIR   = Path('/content/drive/MyDrive/IPD/models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

assert DRIVE_DATA.exists(), f'Drive path not found: {DRIVE_DATA}'

if LOCAL_ROOT.exists() and any(LOCAL_ROOT.iterdir()):
    print('Dataset already on SSD — skipping copy.')
else:
    ZIP_PATH = Path('/content/drive/MyDrive/IPD/dataset.zip')
    if ZIP_PATH.exists():
        import subprocess
        LOCAL_ROOT.mkdir(exist_ok=True)
        print('Copying zip to SSD...')
        subprocess.run(['cp', str(ZIP_PATH), '/content/dataset.zip'], check=True)
        subprocess.run(['unzip', '-q', '/content/dataset.zip', '-d', '/content/'], check=True)
        os.remove('/content/dataset.zip')
        print('Done.')
    else:
        print('Copying dataset from Drive → SSD...')
        shutil.copytree(str(DRIVE_DATA), str(LOCAL_ROOT))
        print('Done.')

for split in ['train', 'val', 'test']:
    d = LOCAL_ROOT / split
    assert d.exists(), f'{split} folder missing'
    n = sum(len(list(c.rglob('*.jpg')) + list(c.rglob('*.jpeg')) + list(c.rglob('*.png')))
            for c in d.iterdir() if c.is_dir())
    print(f'{split:>5} | {n} images')

## 4. Config
Tightened vs 9-class version for smaller dataset (14.5k)

In [ ]:
CFG = dict(
    model_name      = 'efficientnet_b1',
    img_size        = 224,
    batch_size      = 16,      # 8GB effective VRAM (10GB slice minus OS/CUDA overhead)
    num_epochs      = 30,
    lr              = 2e-4,
    weight_decay    = 2e-4,
    dropout         = 0.35,
    drop_path_rate  = 0.15,
    label_smooth    = 0.1,
    warmup_epochs   = 3,
    use_amp         = True,
    num_workers     = 1,       # 1 CPU core available
    freeze_epochs   = 6,
    patience        = 6,
    lr_decay_rate   = 0.1,
    melanoma_boost  = 2.0,
    bcc_boost       = 2.0,
    tinea_boost     = 3.0,
    eczema_boost    = 1.5,
    ckpt_path       = '/content/best_efficientnet_b1_5cls.pth',
)
print('Config ready.')
print(f'  Model        : {CFG["model_name"]}')
print(f'  Batch size   : {CFG["batch_size"]}  (8GB effective VRAM)')
print(f'  num_workers  : {CFG["num_workers"]}  (1 CPU core)')
print(f'  Dropout      : {CFG["dropout"]}')
print(f'  Weight decay : {CFG["weight_decay"]}')
print(f'  Drop path    : {CFG["drop_path_rate"]}')
print(f'  Freeze epochs: {CFG["freeze_epochs"]}')
print(f'  Patience     : {CFG["patience"]}')

## 5. Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((CFG['img_size'] + 32, CFG['img_size'] + 32)),
    transforms.RandomCrop(CFG['img_size']),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.4, contrast=0.4,
        saturation=0.3, hue=0.1
    ),
    transforms.RandomAffine(
        degrees=0, translate=(0.1, 0.1),
        shear=10, scale=(0.85, 1.15)
    ),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
])

val_tf = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
print('Transforms ready.')

## 6. Dataset & DataLoaders

In [ ]:
TRAIN_DIR = LOCAL_ROOT / 'train'
VAL_DIR   = LOCAL_ROOT / 'val'
TEST_DIR  = LOCAL_ROOT / 'test'

train_ds = ImageFolder(TRAIN_DIR, transform=train_tf)
val_ds   = ImageFolder(VAL_DIR,   transform=val_tf)
test_ds  = ImageFolder(TEST_DIR,  transform=val_tf)

CLASSES = train_ds.classes
NUM_CLS  = len(CLASSES)
print(f'Classes ({NUM_CLS}): {CLASSES}')
print(f'Train : {len(train_ds)} | Val : {len(val_ds)} | Test : {len(test_ds)}')

class_counts = Counter(train_ds.targets)
print('\nTraining class distribution:')
for i, cls in enumerate(CLASSES):
    bar = '█' * (class_counts[i] // 100)
    print(f'  {cls:<25} {class_counts[i]:>5}  {bar}')

# Verify expected classes are present
EXPECTED = {'Melanoma', 'Basal_Cell_Carcinoma', 'Tinea', 'Eczema', 'Keratosis'}
found     = set(CLASSES)
assert found == EXPECTED, f'Class mismatch!\n  Expected: {EXPECTED}\n  Found   : {found}'
print('\n✓ All 5 expected classes confirmed.')

class_weights_sampler = 1.0 / torch.tensor(
    [class_counts[i] for i in range(NUM_CLS)], dtype=torch.float
)
sample_w = [class_weights_sampler[l] for l in train_ds.targets]
sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler,
                          num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
print('DataLoaders ready.')

## 7. Model — EfficientNet-B1

In [ ]:
class EfficientSkinClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.35,
                 drop_path_rate=0.15, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b1',
            pretrained     = pretrained,
            num_classes    = 0,
            drop_rate      = dropout,
            drop_path_rate = drop_path_rate,
        )
        feat_dim = self.backbone.num_features  # 1280
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim),            # LayerNorm: no batch-size constraint (was BatchNorm1d)
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 256),
            nn.GELU(),                         # GELU: consistent with ConvNeXt/Swin heads (was SiLU)
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes),
        )
        print(f'EfficientNet-B1 | features={feat_dim} | '
              f'classes={num_classes} | dropout={dropout} | '
              f'drop_path={drop_path_rate}')

    def forward(self, x):
        return self.classifier(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  Backbone frozen    | trainable: {n:,}')

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  Backbone unfrozen  | trainable: {n:,}')


model = EfficientSkinClassifier(
    num_classes    = NUM_CLS,
    dropout        = CFG['dropout'],
    drop_path_rate = CFG['drop_path_rate'],
    pretrained     = True
).to(DEVICE)
model = model.to(memory_format=torch.channels_last)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')

## 8. Loss, Optimiser & Scheduler
Clinical + imbalance aware loss weights:
- Melanoma + BCC boosted for high recall (malignant)
- Tinea boosted for small class size (1150 images)

In [ ]:
# ── Class-weighted loss ─────────────────────────────────────────────────
counts_arr   = np.array([class_counts[i] for i in range(NUM_CLS)], dtype=np.float32)
loss_weights = 1.0 / np.sqrt(counts_arr)
loss_weights /= loss_weights.sum()

# Apply clinical + imbalance boosts
melanoma_idx = CLASSES.index('Melanoma')
bcc_idx      = CLASSES.index('Basal_Cell_Carcinoma')
tinea_idx    = CLASSES.index('Tinea')
eczema_idx   = CLASSES.index('Eczema')

loss_weights[melanoma_idx] *= CFG['melanoma_boost']
loss_weights[bcc_idx]      *= CFG['bcc_boost']
loss_weights[tinea_idx]    *= CFG['tinea_boost']
loss_weights[eczema_idx]   *= CFG['eczema_boost']
loss_weights /= loss_weights.sum()

weights_tensor = torch.tensor(loss_weights, dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight          = weights_tensor,
    label_smoothing = CFG['label_smooth']
)

print('Class weights for loss:')
for i, cls in enumerate(CLASSES):
    tags = []
    if i == melanoma_idx: tags.append('CRITICAL ← 2x')
    if i == bcc_idx:      tags.append('CRITICAL ← 2x')
    if i == tinea_idx:    tags.append('small class ← 3x')
    if i == eczema_idx:   tags.append('small class ← 1.5x')
    tag = '  ' + ' | '.join(tags) if tags else ''
    print(f'  {cls:<25} {loss_weights[i]:.4f}{tag}')

# Layer-wise LR decay
backbone_params = [p for n, p in model.named_parameters()
                   if 'classifier' not in n and p.requires_grad]
head_params     = [p for n, p in model.named_parameters()
                   if 'classifier' in n and p.requires_grad]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr'] * CFG['lr_decay_rate']},
    {'params': head_params,     'lr': CFG['lr']},
], weight_decay=CFG['weight_decay'])

print(f'\nLayer-wise LR: backbone={CFG["lr"]*CFG["lr_decay_rate"]:.2e} | head={CFG["lr"]:.2e}')

def lr_lambda(epoch):
    if epoch < CFG['warmup_epochs']:
        return (epoch + 1) / CFG['warmup_epochs']
    progress = (epoch - CFG['warmup_epochs']) / max(1, CFG['num_epochs'] - CFG['warmup_epochs'])
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler(enabled=CFG['use_amp'])
print('Optimiser / Scheduler ready.')

## 9. Training Loop

In [ ]:
def train_epoch(loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=CFG['use_amp']):
            logits = model(imgs)
            loss   = criterion(logits, lbls)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def val_epoch(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_p, all_l = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=CFG['use_amp']):
            logits = model(imgs)
            loss   = criterion(logits, lbls)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        total      += imgs.size(0)
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_l.extend(lbls.cpu().numpy())
    all_p = np.array(all_p); all_l = np.array(all_l)
    mel_tp = ((all_p == melanoma_idx) & (all_l == melanoma_idx)).sum()
    mel_fn = ((all_p != melanoma_idx) & (all_l == melanoma_idx)).sum()
    bcc_tp = ((all_p == bcc_idx) & (all_l == bcc_idx)).sum()
    bcc_fn = ((all_p != bcc_idx) & (all_l == bcc_idx)).sum()
    mel_sens = mel_tp / (mel_tp + mel_fn) if (mel_tp + mel_fn) > 0 else 0.0
    bcc_sens = bcc_tp / (bcc_tp + bcc_fn) if (bcc_tp + bcc_fn) > 0 else 0.0
    return total_loss / total, correct / total, mel_sens, bcc_sens


print('Sanity check...')
imgs_s, lbls_s = next(iter(train_loader))
print(f'Batch: {imgs_s.shape} | Labels: {lbls_s.shape}')
del imgs_s, lbls_s

history = {
    'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
    'gap': [], 'melanoma_sensitivity': [], 'bcc_sensitivity': [], 'composite': [],
}
best_composite, patience_cnt = 0.0, 0

print(f'\n{"Ep":>4} {"TrLoss":>8} {"VlLoss":>8} {"TrAcc":>7} {"VlAcc":>7} {"MelSens":>8} {"BCCSens":>8} {"Comp":>7} {"LR":>9}')
print('-' * 82)

for epoch in range(1, CFG['num_epochs'] + 1):
    if epoch == 1:
        model.freeze_backbone()
    elif epoch == CFG['freeze_epochs'] + 1:
        model.unfreeze_backbone()
        backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
        head_params     = [p for n, p in model.named_parameters() if 'classifier' in n]
        optimizer = optim.AdamW([
            {'params': backbone_params, 'lr': CFG['lr'] * CFG['lr_decay_rate']},
            {'params': head_params,     'lr': CFG['lr']},
        ], weight_decay=CFG['weight_decay'])
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        print('  → Full fine-tuning with layer-wise LR.')

    tr_loss, tr_acc             = train_epoch(train_loader)
    vl_loss, vl_acc, mel_s, bcc_s = val_epoch(val_loader)
    scheduler.step()

    # Composite checkpoint score: prioritise malignant recall over overall accuracy
    composite = 0.4 * mel_s + 0.3 * bcc_s + 0.3 * vl_acc

    gap = tr_acc - vl_acc
    history['train_loss'].append(tr_loss);         history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);           history['val_acc'].append(vl_acc)
    history['gap'].append(gap)
    history['melanoma_sensitivity'].append(mel_s); history['bcc_sensitivity'].append(bcc_s)
    history['composite'].append(composite)

    tag = ''
    if composite > best_composite:
        best_composite = composite
        torch.save(model.state_dict(), CFG['ckpt_path'])
        patience_cnt = 0; tag = ' ✓'
    else:
        patience_cnt += 1

    current_lr = optimizer.param_groups[-1]['lr']
    gap_flag   = ' ⚠' if gap > 0.10 else (' △' if gap > 0.05 else '')
    print(f'{epoch:>4} {tr_loss:>8.4f} {vl_loss:>8.4f} '
          f'{tr_acc*100:>6.2f}% {vl_acc*100:>6.2f}% '
          f'{mel_s*100:>7.2f}% {bcc_s*100:>7.2f}% '
          f'{composite:>7.4f} {current_lr:>9.2e}{gap_flag}{tag}')

    if patience_cnt >= CFG['patience']:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

best_mel_sens = max(history['melanoma_sensitivity'])
best_val_acc  = max(history['val_acc'])
print(f'\nBest composite   : {best_composite:.4f}')
print(f'Best Val Accuracy: {best_val_acc*100:.2f}%')
print(f'Best Mel Sens    : {best_mel_sens*100:.2f}%  (target ≥ 95%)')
print(f'Avg gap (last 5ep): {np.mean(history["gap"][-5:])*100:.1f}%  (target < 5%)')

## 10. Training Curves + Overfit Monitor

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'],   label='Val',   linewidth=2)
axes[0].axvline(CFG['freeze_epochs'], color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([a*100 for a in history['train_acc']], label='Train', linewidth=2)
axes[1].plot([a*100 for a in history['val_acc']],   label='Val',   linewidth=2)
axes[1].axvline(CFG['freeze_epochs'], color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].set_ylabel('%')
axes[1].legend(); axes[1].grid(alpha=0.3)

gaps = [g*100 for g in history['gap']]
axes[2].plot(gaps, linewidth=2, color='tomato', label='Train-Val gap')
axes[2].axhline(5,  color='orange', linestyle='--', alpha=0.8, label='5% warning')
axes[2].axhline(10, color='red',    linestyle='--', alpha=0.8, label='10% overfit')
axes[2].fill_between(range(len(gaps)), gaps, alpha=0.15, color='tomato')
axes[2].set_title('Overfit Monitor', fontweight='bold'); axes[2].set_ylabel('Gap %')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'EfficientNet-B1 (5-class) | Best Val: {best_val_acc*100:.2f}%',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'efficientnet_b1_5cls_curves.png'), dpi=150)
plt.show()

## 11. Evaluation — Clinical Report
Melanoma + BCC target: 95% and 90% recall
Tinea + Eczema + Keratosis target: 80% recall

In [ ]:
model.load_state_dict(torch.load(CFG['ckpt_path'], map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs   = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)
        logits = model(imgs)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(lbls.numpy()); all_probs.extend(probs)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

test_acc  = accuracy_score(all_labels, all_preds)
y_bin     = label_binarize(all_labels, classes=list(range(NUM_CLS)))
macro_auc = roc_auc_score(y_bin, all_probs, multi_class='ovr', average='macro')

print(f'Val  Accuracy : {best_val_acc*100:.2f}%')
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Macro AUC     : {macro_auc:.4f}')
print('\n' + classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

# Clinical targets per class
CLINICAL_TARGETS = {
    'Melanoma'           : 0.95,   # highest — most deadly
    'Basal_Cell_Carcinoma': 0.90,  # second most dangerous
    'Tinea'              : 0.80,
    'Eczema'             : 0.80,
    'Keratosis'          : 0.80,
}

print('='*68)
print(f'{"Class":<25} {"Sens":>6} {"Spec":>6} {"AUC":>6} {"Target":>7} {"Status":>8}')
print('='*68)
for i, cls in enumerate(CLASSES):
    tp = ((all_preds==i)&(all_labels==i)).sum()
    fn = ((all_preds!=i)&(all_labels==i)).sum()
    tn = ((all_preds!=i)&(all_labels!=i)).sum()
    fp = ((all_preds==i)&(all_labels!=i)).sum()
    sens      = tp/(tp+fn) if (tp+fn)>0 else 0
    spec      = tn/(tn+fp) if (tn+fp)>0 else 0
    auc_score = roc_auc_score(y_bin[:,i], all_probs[:,i])
    target    = CLINICAL_TARGETS[cls]
    is_malig  = cls in ['Melanoma', 'Basal_Cell_Carcinoma']
    flag      = '⚠️' if is_malig else '  '
    status    = '✅' if sens >= target else '❌'
    print(f'{cls:<25} {sens:>6.3f} {spec:>6.3f} {auc_score:>6.3f} '
          f'{target:>6.0%} {flag}{status}')
print('='*68)

# NPV for malignant classes
print('\nNegative Predictive Value (malignant classes):')
for cls in ['Melanoma', 'Basal_Cell_Carcinoma']:
    i  = CLASSES.index(cls)
    tn = ((all_preds!=i)&(all_labels!=i)).sum()
    fn = ((all_preds!=i)&(all_labels==i)).sum()
    npv = tn/(tn+fn) if (tn+fn)>0 else 0
    status = '✅' if npv >= 0.95 else '❌'
    print(f'  {cls:<25} NPV: {npv:.3f}  (target ≥ 0.95) {status}')

## 12. ROC Curves

In [ ]:
plt.figure(figsize=(9, 6))
colors = plt.cm.tab10.colors
for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, lw=2, color=colors[i],
             label=f'{cls}  (AUC={auc(fpr,tpr):.3f})')
plt.plot([0,1],[0,1],'k--',lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — EfficientNet-B1 (5-class)', fontweight='bold')
plt.legend(loc='lower right', fontsize=9); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'efficientnet_b1_5cls_roc.png'), dpi=150)
plt.show()

## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12); ax.set_ylabel('True', fontsize=12)
ax.set_title('EfficientNet-B1 Confusion Matrix (5-class)', fontsize=13)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'efficientnet_b1_5cls_confusion.png'), dpi=150)
plt.show()

## 14. Save

In [ ]:
save_path = SAVE_DIR / 'efficientnet_b1_5cls_skin.pth'
torch.save({
    'model_state'          : model.state_dict(),
    'classes'              : CLASSES,
    'config'               : CFG,
    'best_composite'       : best_composite,
    'best_val_acc'         : best_val_acc,
    'best_melanoma_sens'   : best_mel_sens,
    'test_acc'             : float(test_acc),
    'macro_auc'            : float(macro_auc),
    'history'              : history,
}, save_path)
print(f'✓ Saved to {save_path}')
print(f'  Val acc      : {best_val_acc*100:.2f}%')
print(f'  Test acc     : {test_acc*100:.2f}%')
print(f'  AUC          : {macro_auc:.4f}')
print(f'  Best Mel Sens: {best_mel_sens*100:.2f}%  (target ≥ 95%)')

## 15. Load into Ensemble

In [ ]:
class EfficientSkinClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.35,
                 drop_path_rate=0.15, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b1', pretrained=pretrained,
            num_classes=0, drop_rate=dropout, drop_path_rate=drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim), nn.Dropout(dropout),
            nn.Linear(feat_dim, 256), nn.GELU(),
            nn.Dropout(dropout/2), nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.backbone(x))

def load_efficientnet_b1(path, device):
    ckpt  = torch.load(path, map_location=device)
    model = EfficientSkinClassifier(
        num_classes=len(ckpt['classes']),
        dropout=ckpt['config']['dropout'],
        drop_path_rate=ckpt['config']['drop_path_rate'],
        pretrained=False
    )
    model.load_state_dict(ckpt['model_state'])
    model = model.to(device).eval()
    mel = ckpt.get('best_melanoma_sens', float('nan'))
    print(f'✓ EfficientNet-B1 | val={ckpt["best_val_acc"]*100:.2f}% '
          f'test={ckpt["test_acc"]*100:.2f}% | mel_sens={mel*100:.1f}%')
    return model

# eff = load_efficientnet_b1('/content/drive/MyDrive/IPD/models/efficientnet_b1_5cls_skin.pth', DEVICE)
print('Load function ready.')